# Run C reference sanity check — generator last-p vs Taillard C++

This notebook checks whether our Run C Python reference matches Taillard's C++ `Reference` value.

Important fix in this version:
- `cluster_tai` files have a 3-value header: `n p d`.
- Taillard's C++ reader expects 5 header values: `n p d dummy dummy`.
- Before running the C++ program, this notebook creates a temporary C++-compatible copy of each instance by rewriting the first line as `n p d 0 0`.

The main check is:

```text
python_ref_include_last_p == cpp_labelled_reference
```

C++ options 0/1/2 are baseline methods after the reference is validated; they are not used to validate the reference.


In [ ]:
# ============================================================
# Control panel
# ============================================================

from pathlib import Path

TM_DIR = Path("/content/drive/MyDrive/TM")

CLUSTER_ZIP_PATH = TM_DIR / "cluster_tai.zip"
CPP_PATH = TM_DIR / "clustering_sphere.cpp"

WORK_DIR = Path("/content/runC_taillard_reference_sanity")
EXTRACT_DIR = WORK_DIR / "instances"
CPP_INSTANCE_DIR = WORK_DIR / "instances_cpp_format"
CPP_BIN = WORK_DIR / "clustering_sphere"

# ID is fixed to 1, as requested.
INSTANCE_ID = 1

# Keep this small for sanity checks.
# You can add more specs once the check is working.
TEST_SPECS = [
    {"d": 2, "p": 20},
    {"d": 3, "p": 40},
    {"d": 4, "p": 20},
    # {"d": 4, "p": 70},  # can be slow; enable later if needed
]

# C++ options:
# 0 = kmedian-like refinement
# 1 = PAM-style, very slow; usually only run on small n
# 2 = hybrid PAM on sample + kmedian-like refinement
OPTIONS_TO_TEST = [0, 2]

CPP_RUNS = 1
CPP_TIMEOUT_S = 300

# Avoid PAM option 1 on large instances.
RUN_OPTION_1_PAM_ONLY_IF_N_LEQ = 400

print("TM_DIR:", TM_DIR)
print("CLUSTER_ZIP_PATH:", CLUSTER_ZIP_PATH)
print("CPP_PATH:", CPP_PATH)


In [ ]:
# ============================================================
# Mount Drive and check required files
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("[info] Drive mount skipped or unavailable:", repr(e))

missing = []
for p in [CLUSTER_ZIP_PATH, CPP_PATH]:
    ok = Path(p).exists()
    print(("OK      " if ok else "MISSING "), p)
    if not ok:
        missing.append(str(p))

if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))


In [ ]:
# ============================================================
# Extract cluster_tai.zip
# ============================================================

import shutil
import zipfile
from pathlib import Path

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
CPP_INSTANCE_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(CLUSTER_ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_DIR)

# Some zips contain a nested root folder. Collect all CSV files recursively.
all_csv = sorted(EXTRACT_DIR.rglob("*.csv"))
print("CSV files found:", len(all_csv))
print("\n".join(str(p.relative_to(EXTRACT_DIR)) for p in all_csv[:10]))


In [ ]:
# ============================================================
# Instance helpers
# ============================================================

import re
import numpy as np
import pandas as pd
from pathlib import Path

NAME_RE = re.compile(r"cluster_tai(?P<n>\d+)_(?P<p>\d+)_(?P<d>\d+)_(?P<id>\d+)\.csv$")


def parse_instance_name(path: Path):
    m = NAME_RE.search(path.name)
    if not m:
        return None
    return {
        "name": path.name,
        "n": int(m.group("n")),
        "p": int(m.group("p")),
        "d": int(m.group("d")),
        "instance_id": int(m.group("id")),
        "path": path,
    }


manifest_rows = []
for pth in all_csv:
    row = parse_instance_name(pth)
    if row is not None:
        manifest_rows.append(row)

manifest_df = pd.DataFrame(manifest_rows).sort_values(["d", "p", "instance_id"]).reset_index(drop=True)
print("Parsed instances:", len(manifest_df))
display(manifest_df.head(20))


def find_instance(d: int, p: int, instance_id: int = 1) -> Path:
    sub = manifest_df[
        (manifest_df["d"] == int(d))
        & (manifest_df["p"] == int(p))
        & (manifest_df["instance_id"] == int(instance_id))
    ]
    if len(sub) == 0:
        raise FileNotFoundError(f"No instance found for d={d}, p={p}, id={instance_id}")
    return Path(sub.iloc[0]["path"])


def read_cluster_tai_instance(path: Path):
    '''
    Reads a cluster_tai CSV/text file.
    Expected first line: n p d
    Remaining rows: coordinates.
    '''
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        first = f.readline().strip()
        toks = first.replace(",", " ").replace(";", " ").split()
        if len(toks) < 3:
            raise ValueError(f"Bad header in {path}: {first!r}")
        n, p, d = map(int, toks[:3])

        data = []
        for line in f:
            line = line.strip()
            if not line:
                continue
            vals = [float(x) for x in line.replace(",", " ").replace(";", " ").split()]
            if len(vals) >= d:
                data.append(vals[:d])

    X = np.asarray(data, dtype=float)

    if X.shape[1] != d:
        raise ValueError(f"Expected d={d}, got X.shape={X.shape} for {path}")

    # In these generated instances, n in the header appears to be the number of rows,
    # including the p generator centers as the last p rows.
    if X.shape[0] != n:
        print(f"[warning] Header n={n}, but loaded {X.shape[0]} rows for {path.name}")

    return X, n, p, d


def make_cpp_compatible_copy(src_path: Path, dst_dir: Path) -> Path:
    '''
    Taillard's C++ reader expects:
        n p dim dummy dummy
    while cluster_tai files usually have:
        n p dim
    This function rewrites only the first line to:
        n p dim 0 0
    and copies the rest unchanged.
    '''
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst_path = dst_dir / src_path.name

    with open(src_path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    if not lines:
        raise ValueError(f"Empty instance file: {src_path}")

    toks = lines[0].strip().replace(",", " ").replace(";", " ").split()
    if len(toks) < 3:
        raise ValueError(f"Bad header in {src_path}: {lines[0]!r}")

    n, p, d = toks[:3]
    lines[0] = f"{n} {p} {d} 0 0\n"

    with open(dst_path, "w", encoding="utf-8") as f:
        f.writelines(lines)

    return dst_path


def radius_volume_cost(X: np.ndarray, centers: np.ndarray, batch_size: int = 2048) -> float:
    '''
    Objective: sum_j radius_j^d.
    Each point is assigned to nearest center by Euclidean distance.
    radius_j is the maximum Euclidean distance among assigned points.
    '''
    X = np.asarray(X, dtype=float)
    centers = np.asarray(centers, dtype=float)
    d = X.shape[1]
    p = centers.shape[0]

    radii2 = np.zeros(p, dtype=float)

    for start in range(0, len(X), batch_size):
        xb = X[start:start + batch_size]
        dist2 = ((xb[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
        assign = np.argmin(dist2, axis=1)
        min_dist2 = dist2[np.arange(len(xb)), assign]

        for j in range(p):
            mask = assign == j
            if np.any(mask):
                val = float(np.max(min_dist2[mask]))
                if val > radii2[j]:
                    radii2[j] = val

    # radius^d = (squared_radius)^(d/2)
    return float(np.sum(radii2 ** (d / 2.0)))


In [ ]:
# ============================================================
# Compute Python generator-last-p reference for id=1 instances
# ============================================================

py_rows = []

for spec in TEST_SPECS:
    path = find_instance(d=spec["d"], p=spec["p"], instance_id=INSTANCE_ID)
    X, n, p, d = read_cluster_tai_instance(path)

    centers_include = X[-p:].copy()
    cost_include = radius_volume_cost(X, centers_include)

    # Alternative sanity version: exclude the last p points as elements.
    # This is not expected to match the C++ Reference if C++ clusters all n rows.
    X_no_centers = X[:-p].copy()
    cost_exclude = radius_volume_cost(X_no_centers, centers_include)

    cpp_path = make_cpp_compatible_copy(path, CPP_INSTANCE_DIR)

    py_rows.append({
        "instance": path.name,
        "n": n,
        "loaded_rows": len(X),
        "p": p,
        "d": d,
        "instance_id": INSTANCE_ID,
        "raw_path": str(path),
        "cpp_compatible_path": str(cpp_path),
        "python_ref_include_last_p": cost_include,
        "python_ref_exclude_last_p": cost_exclude,
    })

py_ref_df = pd.DataFrame(py_rows)
display(py_ref_df)


In [ ]:
# ============================================================
# Compile Taillard C++
# ============================================================

import subprocess
from pathlib import Path

CPP_BIN.parent.mkdir(parents=True, exist_ok=True)

compile_cmd = [
    "g++",
    "-O3",
    "-std=c++17",
    str(CPP_PATH),
    "-o",
    str(CPP_BIN),
]

print("Compile command:", " ".join(compile_cmd))
res = subprocess.run(compile_cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

print("returncode:", res.returncode)
if res.stdout.strip():
    print("--- stdout ---")
    print(res.stdout)
if res.stderr.strip():
    print("--- stderr ---")
    print(res.stderr)

if res.returncode != 0:
    raise RuntimeError("C++ compilation failed")

print("Compiled binary:", CPP_BIN)


In [ ]:
# ============================================================
# Run Taillard C++ and parse/reference-check output
# ============================================================

import re
import time
import subprocess
import numpy as np
import pandas as pd
from pathlib import Path

FLOAT_RE = re.compile(r"[-+]?(?:\d+\.\d*|\.\d+|\d+)(?:[eE][-+]?\d+)?")


def ensure_text(x):
    '''subprocess.TimeoutExpired can carry bytes even when text=True; normalize safely.'''
    if x is None:
        return ""
    if isinstance(x, bytes):
        return x.decode("utf-8", errors="replace")
    return str(x)


def parse_all_floats(text: str):
    text = ensure_text(text)
    vals = []
    for m in FLOAT_RE.finditer(text):
        try:
            vals.append(float(m.group(0)))
        except Exception:
            pass
    return vals


def parse_labelled_reference(text: str):
    '''
    Parse Taillard's line:
      Reference (value, time[s]): <value> <time>
    '''
    text = ensure_text(text)
    patterns = [
        r"(?i)Reference\s*\(value,\s*time\[s\]\)\s*:\s*(" + FLOAT_RE.pattern + r")",
        r"(?i)\breference\b[^0-9eE+\-]*(" + FLOAT_RE.pattern + r")",
        r"(?i)\bref(?:erence)?[_ ]?cost\b[^0-9eE+\-]*(" + FLOAT_RE.pattern + r")",
    ]
    for pat in patterns:
        m = re.search(pat, text)
        if m:
            return float(m.group(1))
    return None


def parse_reference_improved_ratio(text: str):
    '''
    Parse line:
      Reference improved with PAM (value/reference, time[s]): <ratio> <time>
    This is not the selected option result; it is an extra reference-improvement line.
    '''
    text = ensure_text(text)
    pat = r"(?i)Reference improved with PAM.*?:\s*(" + FLOAT_RE.pattern + r")\s+(" + FLOAT_RE.pattern + r")"
    m = re.search(pat, text)
    if not m:
        return None, None
    return float(m.group(1)), float(m.group(2))


def parse_final_method_ratio(text: str):
    '''
    Taillard's program prints the selected option result as a final line like:
      <value/reference> <time>
    We parse the last non-empty line containing exactly two floats and not starting with Reference.
    '''
    text = ensure_text(text)
    candidates = []
    for line in text.splitlines():
        stripped = line.strip()
        if not stripped:
            continue
        if stripped.lower().startswith("reference"):
            continue
        vals = parse_all_floats(stripped)
        if len(vals) == 2:
            candidates.append((vals[0], vals[1], stripped))
    if not candidates:
        return None, None, None

    ratio, runtime_s, raw_line = candidates[-1]
    return float(ratio), float(runtime_s), raw_line


def run_cpp(instance_path: Path, option: int, runs: int, timeout_s: int):
    cmd = [str(CPP_BIN), str(instance_path), str(option), str(runs)]
    t0 = time.perf_counter()

    try:
        res = subprocess.run(
            cmd,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            timeout=timeout_s,
        )
        elapsed = time.perf_counter() - t0
        return {
            "returncode": res.returncode,
            "elapsed_s": elapsed,
            "stdout": ensure_text(res.stdout),
            "stderr": ensure_text(res.stderr),
            "timeout": False,
            "cmd": " ".join(cmd),
        }

    except subprocess.TimeoutExpired as e:
        elapsed = time.perf_counter() - t0
        return {
            "returncode": None,
            "elapsed_s": elapsed,
            "stdout": ensure_text(e.stdout),
            "stderr": ensure_text(e.stderr),
            "timeout": True,
            "cmd": " ".join(cmd),
        }


cpp_rows = []

for _, ref_row in py_ref_df.iterrows():
    inst_name = ref_row["instance"]
    n = int(ref_row["n"])
    cpp_instance_path = Path(ref_row["cpp_compatible_path"])
    py_ref = float(ref_row["python_ref_include_last_p"])

    for option in OPTIONS_TO_TEST:
        if option == 1 and n > RUN_OPTION_1_PAM_ONLY_IF_N_LEQ:
            print(f"Skipping option 1/PAM for {inst_name} because n={n} > {RUN_OPTION_1_PAM_ONLY_IF_N_LEQ}")
            continue

        print("\n" + "=" * 100)
        print(f"Running C++: instance={inst_name}, option={option}, runs={CPP_RUNS}")

        result = run_cpp(
            instance_path=cpp_instance_path,
            option=option,
            runs=CPP_RUNS,
            timeout_s=CPP_TIMEOUT_S,
        )

        combined_output = ensure_text(result["stdout"]) + "\n" + ensure_text(result["stderr"])

        print("Command:", result["cmd"])
        print(
            "Timeout:", result["timeout"],
            "returncode:", result["returncode"],
            "elapsed_s:", round(result["elapsed_s"], 3),
        )

        if result["stdout"].strip():
            print("--- stdout ---")
            print(result["stdout"][:4000])

        if result["stderr"].strip():
            print("--- stderr ---")
            print(result["stderr"][:2000])

        labelled_ref = parse_labelled_reference(combined_output)
        pam_ref_ratio, pam_ref_runtime = parse_reference_improved_ratio(combined_output)
        method_ratio, method_runtime, method_raw_line = parse_final_method_ratio(combined_output)

        cpp_ref_rel_diff = np.nan
        if labelled_ref is not None:
            cpp_ref_rel_diff = abs(labelled_ref - py_ref) / max(1.0, abs(py_ref))

        method_abs_from_cpp_ratio = np.nan
        method_gap_vs_python_ref_pct = np.nan

        if labelled_ref is not None and method_ratio is not None:
            method_abs_from_cpp_ratio = method_ratio * labelled_ref
            method_gap_vs_python_ref_pct = 100.0 * (method_abs_from_cpp_ratio / py_ref - 1.0)

        cpp_rows.append({
            "instance": inst_name,
            "n": n,
            "p": int(ref_row["p"]),
            "d": int(ref_row["d"]),
            "option": option,
            "runs": CPP_RUNS,
            "timeout": result["timeout"],
            "returncode": result["returncode"],
            "elapsed_s": result["elapsed_s"],
            "python_ref_include_last_p": py_ref,
            "python_ref_exclude_last_p": float(ref_row["python_ref_exclude_last_p"]),
            "cpp_labelled_reference": labelled_ref,
            "cpp_ref_rel_diff_vs_python": cpp_ref_rel_diff,
            "cpp_reference_matches_python": bool(cpp_ref_rel_diff <= 1e-9) if not pd.isna(cpp_ref_rel_diff) else False,
            "cpp_pam_improved_reference_ratio": pam_ref_ratio,
            "cpp_pam_improved_reference_runtime_s": pam_ref_runtime,
            "cpp_method_ratio": method_ratio,
            "cpp_method_runtime_s": method_runtime,
            "cpp_method_raw_line": method_raw_line,
            "cpp_method_abs_cost_from_ratio": method_abs_from_cpp_ratio,
            "cpp_method_gap_vs_python_ref_pct": method_gap_vs_python_ref_pct,
            "stdout_first_1000": result["stdout"][:1000],
            "stderr_first_1000": result["stderr"][:1000],
        })

        # Save partial results after each run, so interruption does not lose everything.
        cpp_df = pd.DataFrame(cpp_rows)

cpp_df = pd.DataFrame(cpp_rows)
display(cpp_df)


In [ ]:
# ============================================================
# Interpret comparison
# ============================================================

if "cpp_df" not in globals():
    if "cpp_rows" in globals() and len(cpp_rows) > 0:
        cpp_df = pd.DataFrame(cpp_rows)
        print(f"Recovered cpp_df from cpp_rows: {len(cpp_df)} rows")
    else:
        raise RuntimeError("No cpp_df/cpp_rows found. Run the C++ comparison cell first.")

summary = cpp_df.copy()

cols = [
    "instance", "n", "p", "d", "option", "timeout", "returncode", "elapsed_s",
    "python_ref_include_last_p",
    "python_ref_exclude_last_p",
    "cpp_labelled_reference",
    "cpp_ref_rel_diff_vs_python",
    "cpp_reference_matches_python",
    "cpp_pam_improved_reference_ratio",
    "cpp_method_ratio",
    "cpp_method_abs_cost_from_ratio",
    "cpp_method_gap_vs_python_ref_pct",
    "cpp_method_runtime_s",
]

existing_cols = [c for c in cols if c in summary.columns]
display(summary[existing_cols])

print("\nInterpretation:")
print("- Main sanity check: cpp_labelled_reference must match python_ref_include_last_p.")
print("- If cpp_reference_matches_python is True, our generator-last-p Run C reference matches Taillard's Reference.")
print("- cpp_method_ratio is the C++ option result: method_cost / C++ Reference.")
print("- cpp_method_gap_vs_python_ref_pct converts that same option result to our gap convention using Python's reference.")
print("- Different C++ options are not used to validate the reference; they are baseline methods once the reference denominator is confirmed.")

bad = summary[
    summary["cpp_labelled_reference"].notna()
    & (summary["cpp_ref_rel_diff_vs_python"] > 1e-9)
]

if len(bad) == 0:
    print("\nOK: all parsed C++ Reference values match the Python last-p reference within tolerance.")
else:
    print("\nWARNING: some C++ Reference values do not match the Python last-p reference:")
    display(
        bad[
            [
                "instance",
                "option",
                "python_ref_include_last_p",
                "cpp_labelled_reference",
                "cpp_ref_rel_diff_vs_python",
            ]
        ]
    )


In [ ]:
# ============================================================
# Save sanity-check results + auto-download to local PC
# ============================================================

import zipfile
from pathlib import Path
import pandas as pd

out_dir = TM_DIR / "llm-clustering-runs" / "runC_reference_sanity_checks"
out_dir.mkdir(parents=True, exist_ok=True)

timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

py_ref_path = out_dir / f"python_generator_reference_{timestamp}.csv"
cpp_cmp_path = out_dir / f"taillard_cpp_reference_comparison_{timestamp}.csv"
zip_path = out_dir / f"runC_reference_sanity_check_{timestamp}.zip"

py_ref_df.to_csv(py_ref_path, index=False)

if "cpp_df" in globals():
    cpp_df.to_csv(cpp_cmp_path, index=False)
else:
    cpp_cmp_path = None

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(py_ref_path, arcname=py_ref_path.name)
    if cpp_cmp_path is not None:
        zf.write(cpp_cmp_path, arcname=cpp_cmp_path.name)

print("Saved to Drive:")
print(py_ref_path)
if cpp_cmp_path is not None:
    print(cpp_cmp_path)
print(zip_path)

try:
    from google.colab import files
    files.download(str(zip_path))
    print("Download triggered:", zip_path.name)
except Exception as e:
    print("[info] Automatic download unavailable:", repr(e))
    print("Manual download path:", zip_path)
